# Portfolio version

This notebook was cleaned for public demonstration. Outputs, generated corpora, machine-specific paths, and private research material are not included. The included CSV is synthetic. Check current access rules and page structure before running any web collection.


# MFA China–Australia Multi-keyword Dynamic Crawler + TOFI Pipeline

This notebook collects China MFA search results for multiple keywords related to China–Australia relations.

Workflow:

1. Install dependencies.
2. Search multiple keywords on the dynamic MFA search page.
3. Use Playwright to render JavaScript-loaded results.
4. Crawl result pages sequentially.
5. Extract valid MFA article links.
6. Download article text.
7. Keep Australia-related passages.
8. Score TOFI and cooperation frames.
9. Export CSV and Excel outputs.

In [ ]:
# ============================================================
# 0. Install dependencies
# ============================================================

!pip -q install playwright requests beautifulsoup4 pandas numpy tqdm jieba openpyxl lxml
!playwright install --with-deps chromium

In [ ]:
# ============================================================
# 1. Project setup
# ============================================================

import os
import re
import json
import time
import asyncio
import random
from urllib.parse import quote, urljoin

import requests
import pandas as pd
import numpy as np
import jieba
from bs4 import BeautifulSoup
from tqdm import tqdm
from playwright.async_api import async_playwright

for d in ["data", "data/mfa_transcripts", "data/corpus", "data/analysis", "output", "debug"]:
    os.makedirs(d, exist_ok=True)

BASE_URL = "https://www.mfa.gov.cn"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
    "Connection": "keep-alive",
}

print("Project folders and dependencies are ready.")

In [ ]:
# ============================================================
# 2. Configuration: keywords, years, and search page settings
# ============================================================

START_YEAR = 1972
END_YEAR = 2026

# First test with 3-5 pages. After it works, increase to 20, 30, or more.
MAX_SEARCH_PAGES_PER_KEYWORD = 50
HEADLESS = True
WAIT_AFTER_PAGE_LOAD_MS = 3500
WAIT_AFTER_CLICK_MS = 2500

# Multi-keyword search. This follows the logic of searching different keywords
# and crawling results page by page in order.
SEARCH_KEYWORDS = [
    "澳大利亚",
    "中澳",
    "澳方",
    "澳洲",
    "堪培拉",
    "AUKUS",
    "澳核潜艇",
    "五眼联盟 澳大利亚",
    "中国 澳大利亚 关系",
]

AUSTRALIA_KEYWORDS = [
    "澳大利亚", "澳方", "中澳", "澳政府", "澳方面",
    "澳洲", "堪培拉", "澳核潜艇", "AUKUS",
    "Australia", "Australian", "Canberra",
]

USEFUL_TITLE_KEYWORDS = [
    "发言人", "例行记者会", "答记者问", "外交部发言人",
    "王毅", "秦刚", "赵立坚", "华春莹", "汪文斌", "毛宁", "林剑",
    "会见", "会谈", "对话", "通话", "访问", "声明",
    "外交与战略对话", "战略经济对话", "中澳关系", "澳大利亚",
    "AUKUS", "核潜艇", "五眼联盟",
]

EXCLUDE_TITLE_KEYWORDS = [
    "领事", "签证", "护照", "提醒", "旅游", "留学服务",
    "驻澳大利亚使馆", "驻悉尼", "驻墨尔本", "驻珀斯", "驻布里斯班",
    "招聘", "招标", "通知", "馆舍", "侨务",
]

print("Keywords configured:")
for k in SEARCH_KEYWORDS:
    print("-", k)

In [ ]:
# ============================================================
# 3. Helper functions
# ============================================================

def clean_text(text):
    text = re.sub(r"\s+", " ", str(text))
    text = text.replace("\u3000", " ")
    return text.strip()

def contains_any(text, keywords):
    text = str(text)
    return any(k in text for k in keywords)

def is_australia_relevant(text):
    return contains_any(text, AUSTRALIA_KEYWORDS)

def build_mfa_search_url(keyword):
    """
    Build the MFA dynamic search URL.
    If the page stops loading in the future, copy the latest sign value
    from the browser search URL and replace it below.
    """
    return (
        "https://www.mfa.gov.cn/irs-c-web/search.shtml?"
        "code=17e50b77dab"
        "&codes="
        "&configCode="
        f"&searchWord={quote(keyword)}"
        "&dataTypeId=2082"
        "&sign=ca8802ad-37ef-4f67-edbd-1b9b754da74c"
    )

def normalize_mfa_url(url):
    if not url:
        return None
    url = str(url).strip()
    if url.startswith("//"):
        url = "https:" + url
    elif url.startswith("/"):
        url = urljoin(BASE_URL, url)
    return url

def is_valid_mfa_article_url(url):
    if not url:
        return False
    url = str(url).strip()

    if "mfa.gov.cn" not in url and "fmprc.gov.cn" not in url:
        return False

    bad_patterns = [
        "search.shtml", "javascript:", "#",
        ".css", ".js", ".png", ".jpg", ".jpeg", ".gif", ".ico", ".svg",
        "lmbf/cs/index.shtml",
    ]
    if any(p in url for p in bad_patterns):
        return False

    if not (url.endswith(".shtml") or url.endswith(".html") or "/t" in url):
        return False

    if url.endswith("/index.shtml") or url.endswith("/index.html"):
        return False

    return True

def is_useful_title(title):
    title = str(title)
    if contains_any(title, EXCLUDE_TITLE_KEYWORDS):
        return False
    return contains_any(title, USEFUL_TITLE_KEYWORDS) or is_australia_relevant(title)

def infer_text_type(title, url):
    title = str(title)
    url = str(url)
    if "例行记者会" in title or "jzhsl" in url:
        return "例行记者会"
    if "答记者问" in title or "发言人" in title:
        return "发言人表态/答记者问"
    if any(k in title for k in ["王毅", "外交部长", "会见", "会谈", "对话", "通话", "访问"]):
        return "外交部活动/会谈通稿"
    if "声明" in title:
        return "声明"
    return "其他官方文本"

def infer_stage(date_str):
    """
    Classify China-Australia relations into three historical stages.

    Stage 1: 1972-2016, relatively stable and positive development
    Stage 2: 2017-2021, deterioration and Cold War-style confrontation
    Stage 3: 2022-2026, adjustment and diplomatic repair
    """
    if not date_str:
        return ""

    try:
        year = int(str(date_str)[:4])
    except Exception:
        return ""

    if 1972 <= year <= 2016:
        return "1972-2016 早期稳定向好"
    elif 2017 <= year <= 2021:
        return "2017-2021 中期恶化冷战"
    elif 2022 <= year <= 2026:
        return "2022-2026 后期重新调整"
    else:
        return "研究时段外"

def infer_event(title, passages=""):
    text = f"{title} {passages}"
    rules = [
        ("AUKUS/核潜艇", ["AUKUS", "核潜艇", "澳核潜艇"]),
        ("五眼联盟/印太安全", ["五眼联盟", "印太", "南海", "安全合作"]),
        ("经贸摩擦/贸易争端", ["贸易", "关税", "制裁", "经济胁迫", "葡萄酒", "大麦", "煤炭"]),
        ("新冠溯源/疫情争议", ["新冠", "溯源", "疫情"]),
        ("人权/涉疆涉港涉台", ["新疆", "香港", "台湾", "人权", "涉疆", "涉港", "涉台"]),
        ("双边关系修复/对话", ["改善", "修复", "重启", "对话", "会谈", "合作", "互利"]),
        ("外国干涉/涉华言论", ["干涉", "内政", "涉华", "反华", "无端指责"]),
    ]
    for label, kws in rules:
        if any(k in text for k in kws):
            return label
    return "其他议题"

In [ ]:
# ============================================================
# 4. Link extraction from rendered DOM and network responses
# ============================================================

def extract_links_from_json_like(obj, keyword="", page_no=None, source="network_json"):
    """
    Recursively extract possible title-url pairs from JSON responses.
    This helps when the search page loads results through an API.
    """
    results = []

    def walk(x):
        if isinstance(x, dict):
            possible_url = None
            possible_title = None

            for key in ["url", "href", "docUrl", "docurl", "link", "pageUrl", "urlPath"]:
                if key in x and isinstance(x[key], str):
                    possible_url = x[key]
                    break

            for key in ["title", "docTitle", "name", "contentTitle", "chnldesc", "subtitle"]:
                if key in x and isinstance(x[key], str):
                    possible_title = x[key]
                    break

            if possible_url:
                url = normalize_mfa_url(possible_url)
                title = clean_text(possible_title or "")
                if is_valid_mfa_article_url(url):
                    results.append({
                        "keyword": keyword,
                        "result_page": page_no,
                        "title": title,
                        "url": url,
                        "link_source": source,
                    })

            for v in x.values():
                walk(v)

        elif isinstance(x, list):
            for item in x:
                walk(item)

    walk(obj)
    return results

async def extract_links_from_rendered_dom(page, keyword="", page_no=None):
    raw_items = await page.evaluate("""
        () => {
            const arr = [];
            const anchors = Array.from(document.querySelectorAll('a[href]'));
            for (const a of anchors) {
                arr.push({
                    title: (a.innerText || a.textContent || '').trim(),
                    url: a.href
                });
            }
            return arr;
        }
    """)

    results = []
    for item in raw_items:
        url = normalize_mfa_url(item.get("url"))
        title = clean_text(item.get("title", ""))

        if not is_valid_mfa_article_url(url):
            continue
        if len(title) < 4:
            continue
        if contains_any(title, EXCLUDE_TITLE_KEYWORDS):
            continue

        results.append({
            "keyword": keyword,
            "result_page": page_no,
            "title": title,
            "url": url,
            "link_source": "rendered_dom",
        })

    return results

async def save_debug_files(page, keyword, page_no):
    safe_kw = re.sub(r"[^\w\u4e00-\u9fff]+", "_", keyword)
    try:
        await page.screenshot(path=f"debug/search_{safe_kw}_page_{page_no}.png", full_page=True)
    except Exception:
        pass
    try:
        html = await page.content()
        with open(f"debug/search_{safe_kw}_page_{page_no}.html", "w", encoding="utf-8") as f:
            f.write(html)
    except Exception:
        pass

In [ ]:
# ============================================================
# 5. Robust next-page clicking
# ============================================================

async def try_click_next_page(page):
    old_url = page.url

    try:
        old_text = await page.locator("body").inner_text(timeout=5000)
    except Exception:
        old_text = ""

    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
    await page.wait_for_timeout(1000)

    selectors = [
        "text=下一页",
        "a:has-text('下一页')",
        "button:has-text('下一页')",
        "span:has-text('下一页')",
        "li:has-text('下一页')",
        "a[title='下一页']",
        "button[title='下一页']",
        ".next",
        ".next-page",
        ".page-next",
        ".btn-next",
        "[class*='next']",
    ]

    for selector in selectors:
        try:
            locator = page.locator(selector)
            count = await locator.count()
            if count == 0:
                continue

            for i in range(count):
                target = locator.nth(i)
                try:
                    if await target.is_visible():
                        await target.click(force=True)
                        await page.wait_for_timeout(WAIT_AFTER_CLICK_MS)

                        try:
                            new_text = await page.locator("body").inner_text(timeout=5000)
                        except Exception:
                            new_text = ""

                        if page.url != old_url or new_text[:500] != old_text[:500]:
                            return True
                except Exception:
                    continue
        except Exception:
            continue

    try:
        clicked = await page.evaluate("""
            () => {
                const candidates = Array.from(document.querySelectorAll('a, button, span, li, div'));
                for (const el of candidates) {
                    const text = (el.innerText || el.textContent || '').trim();
                    const cls = String(el.className || '').toLowerCase();
                    const title = String(el.getAttribute('title') || '');
                    const aria = String(el.getAttribute('aria-label') || '');

                    const looksNext =
                        text.includes('下一页') ||
                        text === '>' ||
                        text === '›' ||
                        title.includes('下一页') ||
                        aria.toLowerCase().includes('next') ||
                        cls.includes('next');

                    const disabled =
                        cls.includes('disabled') ||
                        el.getAttribute('disabled') !== null ||
                        el.getAttribute('aria-disabled') === 'true';

                    if (looksNext && !disabled) {
                        el.click();
                        return true;
                    }
                }
                return false;
            }
        """)

        if clicked:
            await page.wait_for_timeout(WAIT_AFTER_CLICK_MS)
            try:
                new_text = await page.locator("body").inner_text(timeout=5000)
            except Exception:
                new_text = ""
            if page.url != old_url or new_text[:500] != old_text[:500]:
                return True
    except Exception:
        pass

    return False

In [ ]:
# ============================================================
# 6. Dynamic crawler: multi-keyword, page-by-page, network-first
# ============================================================

async def extract_current_page_results(page, keyword, page_no):
    """
    Extract visible search results from the current rendered page.
    This works after JavaScript has loaded the result list.
    """
    items = await page.evaluate("""
        () => {
            const results = [];
            const anchors = Array.from(document.querySelectorAll('a[href]'));

            for (const a of anchors) {
                const title = (a.innerText || a.textContent || '').trim();
                const href = a.href;

                if (!href || !title) continue;

                results.push({
                    title: title,
                    url: href
                });
            }

            return results;
        }
    """)

    cleaned = []

    for item in items:
        title = clean_text(item.get("title", ""))
        url = normalize_mfa_url(item.get("url", ""))

        # 去掉搜索页、导航页、图片、index 页
        if not is_valid_mfa_article_url(url):
            continue

        # 去掉太短的导航标题
        if len(title) < 4:
            continue

        # 去掉明显无关的服务/领事类页面
        if contains_any(title, EXCLUDE_TITLE_KEYWORDS):
            continue

        cleaned.append({
            "keyword": keyword,
            "result_page": page_no,
            "title": title,
            "url": url,
            "link_source": "rendered_dom",
        })

    return cleaned


async def wait_until_results_change(page, old_signature, timeout_ms=8000):
    """
    Wait until the first few result titles/URLs change after pagination.
    """
    start = time.time()

    while (time.time() - start) * 1000 < timeout_ms:
        await page.wait_for_timeout(800)

        try:
            new_items = await page.evaluate("""
                () => {
                    const anchors = Array.from(document.querySelectorAll('a[href]'));
                    return anchors.slice(0, 20).map(a => {
                        return ((a.innerText || a.textContent || '').trim()) + '|' + a.href;
                    }).join('\\n');
                }
            """)
        except Exception:
            new_items = ""

        if new_items and new_items != old_signature:
            return True

    return False


async def get_page_signature(page):
    try:
        sig = await page.evaluate("""
            () => {
                const anchors = Array.from(document.querySelectorAll('a[href]'));
                return anchors.slice(0, 20).map(a => {
                    return ((a.innerText || a.textContent || '').trim()) + '|' + a.href;
                }).join('\\n');
            }
        """)
        return sig
    except Exception:
        return ""


async def click_next_page_more_robust(page):
    """
    More robust pagination clicker for MFA search page.
    It tries normal selectors, text, arrow symbols, class names, and JS click.
    """
    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
    await page.wait_for_timeout(1200)

    old_signature = await get_page_signature(page)

    # 1. 常见“下一页”选择器
    selectors = [
        "text=下一页",
        "a:has-text('下一页')",
        "button:has-text('下一页')",
        "span:has-text('下一页')",
        "li:has-text('下一页')",
        "div:has-text('下一页')",
        "a[title='下一页']",
        "button[title='下一页']",
        ".next",
        ".page-next",
        ".next-page",
        ".btn-next",
        "[class*='next']",
    ]

    for selector in selectors:
        try:
            loc = page.locator(selector)
            count = await loc.count()

            if count == 0:
                continue

            for i in range(count):
                el = loc.nth(i)

                try:
                    if await el.is_visible():
                        await el.click(force=True)
                        changed = await wait_until_results_change(page, old_signature)

                        if changed:
                            return True
                except Exception:
                    continue

        except Exception:
            continue

    # 2. JS 查找下一页按钮，包括 >、›、下一页、next class
    try:
        clicked = await page.evaluate("""
            () => {
                const candidates = Array.from(document.querySelectorAll('a, button, span, li, div'));

                for (const el of candidates) {
                    const text = (el.innerText || el.textContent || '').trim();
                    const title = el.getAttribute('title') || '';
                    const cls = String(el.className || '').toLowerCase();
                    const aria = el.getAttribute('aria-label') || '';

                    const looksNext =
                        text === '下一页' ||
                        text.includes('下一页') ||
                        text === '>' ||
                        text === '›' ||
                        title.includes('下一页') ||
                        aria.toLowerCase().includes('next') ||
                        cls.includes('next');

                    const disabled =
                        cls.includes('disabled') ||
                        el.getAttribute('disabled') !== null ||
                        el.getAttribute('aria-disabled') === 'true';

                    if (looksNext && !disabled) {
                        el.scrollIntoView();
                        el.click();
                        return true;
                    }
                }

                return false;
            }
        """)

        if clicked:
            changed = await wait_until_results_change(page, old_signature)
            if changed:
                return True

    except Exception:
        pass

    # 3. 兜底：键盘 PageDown + Tab/Enter，适用于分页按钮无法被 selector 抓到的情况
    try:
        await page.keyboard.press("End")
        await page.wait_for_timeout(800)
        await page.keyboard.press("Tab")
        await page.wait_for_timeout(300)
        await page.keyboard.press("Enter")

        changed = await wait_until_results_change(page, old_signature)

        if changed:
            return True

    except Exception:
        pass

    return False


async def collect_links_for_keyword(keyword, max_pages=30, headless=True):
    search_url = build_mfa_search_url(keyword)

    print(f"\n========== Keyword: {keyword} ==========")
    print(search_url)

    all_items = []
    seen_urls = set()
    seen_page_signatures = set()
    network_buffer = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=headless,
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu",
                "--disable-blink-features=AutomationControlled",
            ],
        )

        context = await browser.new_context(
            viewport={"width": 1400, "height": 1000},
            user_agent=HEADERS["User-Agent"],
            locale="zh-CN",
        )

        page = await context.new_page()

        async def handle_response(response):
            """
            Capture JSON responses from the dynamic search system.
            """
            try:
                resp_url = response.url
                ctype = response.headers.get("content-type", "")

                if "irs" not in resp_url and "search" not in resp_url:
                    return

                if "json" in ctype.lower():
                    data = await response.json()
                    extracted = extract_links_from_json_like(
                        data,
                        keyword=keyword,
                        page_no=None,
                        source="network_json",
                    )
                    network_buffer.extend(extracted)

            except Exception:
                return

        page.on("response", handle_response)

        await page.goto(search_url, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(WAIT_AFTER_PAGE_LOAD_MS)

        for page_no in range(1, max_pages + 1):
            print(f"Collecting page {page_no} for keyword [{keyword}]...")

            # 等待当前页结果加载
            await page.wait_for_timeout(2500)

            # 滚动，触发动态元素完整渲染
            for _ in range(3):
                await page.mouse.wheel(0, 1000)
                await page.wait_for_timeout(600)

            # 每隔几页保存 debug
            if page_no == 1 or page_no % 10 == 0:
                await save_debug_files(page, keyword, page_no)

            dom_items = await extract_current_page_results(page, keyword, page_no)

            # 把 network 抓到的链接也放进来
            network_items = []
            if network_buffer:
                for item in network_buffer:
                    item = dict(item)
                    item["result_page"] = page_no
                    network_items.append(item)
                network_buffer.clear()

            current_items = dom_items + network_items

            # 当前页去重
            page_unique = {}
            for item in current_items:
                url = item.get("url", "")
                title = clean_text(item.get("title", ""))

                if not is_valid_mfa_article_url(url):
                    continue

                if not title:
                    title = ""

                item["title"] = title

                page_unique[url] = item

            current_items = list(page_unique.values())

            # 防止重复页面
            page_signature = tuple(sorted([x["url"] for x in current_items])[:8])

            if page_signature and page_signature in seen_page_signatures:
                print("  Repeated page detected. Stop this keyword.")
                break

            if page_signature:
                seen_page_signatures.add(page_signature)

            new_count = 0
            for item in current_items:
                url = item["url"]
                if url not in seen_urls:
                    seen_urls.add(url)
                    all_items.append(item)
                    new_count += 1

            print(f"  Current page links: {len(current_items)}; new links: {new_count}; total: {len(all_items)}")

            # 如果这一页完全没有新链接，尝试下一页，但不要无限循环
            clicked = await click_next_page_more_robust(page)

            if not clicked:
                print("  Could not move to next page. Stop this keyword.")
                break

        await browser.close()

    print(f"Keyword [{keyword}] finished: {len(all_items)} unique links.")
    return all_items


async def collect_links_for_all_keywords(keywords, max_pages=30, headless=True):
    all_items = []

    for kw in keywords:
        try:
            items = await collect_links_for_keyword(
                kw,
                max_pages=max_pages,
                headless=headless,
            )
            all_items.extend(items)

            # 礼貌等待，避免访问过快
            await asyncio.sleep(random.uniform(2, 4))

        except Exception as e:
            print(f"ERROR while crawling keyword [{kw}]: {repr(e)}")
            continue

    # 全局按 URL 去重，并合并关键词
    by_url = {}

    for item in all_items:
        url = item.get("url", "")

        if not url:
            continue

        if url not in by_url:
            by_url[url] = item
        else:
            old_kw = str(by_url[url].get("keyword", ""))
            new_kw = str(item.get("keyword", ""))

            kws = set(old_kw.split(";"))
            kws.add(new_kw)

            by_url[url]["keyword"] = ";".join([k for k in kws if k])

    return list(by_url.values())


# ============================================================
# Run crawler in Colab
# ============================================================

candidate_links = await collect_links_for_all_keywords(
    SEARCH_KEYWORDS,
    max_pages=MAX_SEARCH_PAGES_PER_KEYWORD,
    headless=HEADLESS,
)

df_links = pd.DataFrame(candidate_links)

if not df_links.empty:
    df_links = df_links.drop_duplicates(subset=["url"]).reset_index(drop=True)

    df_links.to_csv(
        "data/mfa_transcripts/mfa_dynamic_search_links_raw.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("\nTotal unique candidate links:", len(df_links))
display(df_links.head(100))
print("Saved: data/mfa_transcripts/mfa_dynamic_search_links_raw.csv")

In [ ]:
# ============================================================
# 7. Further filter candidate links
# ============================================================

if df_links.empty:
    raise ValueError(
        "No links were collected. Check debug screenshots/html in the debug folder. "
        "The MFA page may have changed, or Colab may be blocked."
    )

def keep_candidate_row(row):
    title = str(row.get("title", ""))
    keyword = str(row.get("keyword", ""))
    url = str(row.get("url", ""))

    if not is_valid_mfa_article_url(url):
        return False
    if contains_any(title, EXCLUDE_TITLE_KEYWORDS):
        return False
    if is_useful_title(title):
        return True
    if is_australia_relevant(title):
        return True
    if any(k in keyword for k in ["澳大利亚", "中澳", "澳方", "AUKUS", "澳核潜艇"]):
        return True
    return False

df_links_filtered = df_links[df_links.apply(keep_candidate_row, axis=1)].copy()
df_links_filtered = df_links_filtered.drop_duplicates(subset=["url"]).reset_index(drop=True)

df_links_filtered.to_csv(
    "data/mfa_transcripts/mfa_links_to_scrape.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Before filtering:", len(df_links))
print("After filtering:", len(df_links_filtered))
display(df_links_filtered.head(50))
print("Saved: data/mfa_transcripts/mfa_links_to_scrape.csv")

In [ ]:
# ============================================================
# 8. Scrape article body and Australia-related passages
# ============================================================

def extract_main_body(soup):
    selectors = [
        "div.TRS_Editor",
        "div.trs_editor",
        "div.news-content",
        "div.article",
        "div.content",
        "div#News_Body_Txt_A",
        "article",
        "main",
    ]

    for sel in selectors:
        body = soup.select_one(sel)
        if body:
            txt = clean_text(body.get_text("\n"))
            if len(txt) > 80:
                return body

    return soup.body

def extract_title(soup, fallback=""):
    selectors = ["h1", "h2", ".title", "title"]
    for sel in selectors:
        node = soup.select_one(sel)
        if node:
            txt = clean_text(node.get_text(" "))
            if len(txt) > 3:
                return txt
    return clean_text(fallback)

def extract_date_from_text_or_url(text, url):
    text = str(text)
    url = str(url)

    patterns = [
        r"(20\d{2})[-年./](\d{1,2})[-月./](\d{1,2})",
        r"(20\d{2})(\d{2})(\d{2})",
    ]

    for pat in patterns:
        m = re.search(pat, text)
        if m:
            y, mo, d = m.group(1), m.group(2), m.group(3)
            return f"{int(y):04d}-{int(mo):02d}-{int(d):02d}"

    m = re.search(r"/(20\d{2})(\d{2})/t(20\d{2})(\d{2})(\d{2})", url)
    if m:
        return f"{int(m.group(3)):04d}-{int(m.group(4)):02d}-{int(m.group(5)):02d}"

    m = re.search(r"t(20\d{2})(\d{2})(\d{2})", url)
    if m:
        return f"{int(m.group(1)):04d}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"

    return ""

def split_paragraphs(raw_text):
    raw_text = str(raw_text).replace("\r", "\n")
    parts = [clean_text(p) for p in raw_text.split("\n") if len(clean_text(p)) > 5]

    if len(parts) < 4:
        parts = re.split(r"(?=问[:：]|答[:：]|记者[:：]|发言人[:：])", clean_text(raw_text))
        parts = [clean_text(p) for p in parts if len(clean_text(p)) > 5]

    return parts

def select_relevant_context(paragraphs, window=1):
    keep_idx = set()
    for i, p in enumerate(paragraphs):
        if is_australia_relevant(p):
            for j in range(i - window, i + window + 1):
                if 0 <= j < len(paragraphs):
                    keep_idx.add(j)
    return [paragraphs[i] for i in sorted(keep_idx)]

def scrape_one_article(row, delay_range=(0.8, 1.8)):
    url = row["url"]
    title_from_link = row.get("title", "")

    try:
        time.sleep(random.uniform(*delay_range))
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.encoding = r.apparent_encoding

        if r.status_code != 200:
            return {
                "ok": False,
                "status_code": r.status_code,
                "url": url,
                "title": title_from_link,
                "error": f"HTTP {r.status_code}",
            }

        soup = BeautifulSoup(r.text, "html.parser")

        title = extract_title(soup, fallback=title_from_link)
        full_text_raw = clean_text(soup.get_text("\n"))
        date = extract_date_from_text_or_url(full_text_raw[:1000], url)

        body_node = extract_main_body(soup)
        body_text = clean_text(body_node.get_text("\n")) if body_node else full_text_raw

        paragraphs = split_paragraphs(body_text)
        relevant_paragraphs = select_relevant_context(paragraphs, window=1)
        relevant_text = "\n".join(relevant_paragraphs)

        article_relevant = is_australia_relevant(title) or is_australia_relevant(body_text)

        return {
            "ok": article_relevant,
            "status_code": r.status_code,
            "keyword": row.get("keyword", ""),
            "result_page": row.get("result_page", ""),
            "link_source": row.get("link_source", ""),
            "url": url,
            "title": title,
            "date": date,
            "year": int(date[:4]) if re.match(r"^\d{4}", str(date)) else np.nan,
            "text_type": infer_text_type(title, url),
            "stage": infer_stage(date),
            "event": infer_event(title, relevant_text),
            "full_text": body_text,
            "australia_passages": relevant_text,
            "n_chars_full": len(body_text),
            "n_chars_passages": len(relevant_text),
            "error": "",
        }

    except Exception as e:
        return {
            "ok": False,
            "status_code": "",
            "keyword": row.get("keyword", ""),
            "result_page": row.get("result_page", ""),
            "link_source": row.get("link_source", ""),
            "url": url,
            "title": title_from_link,
            "date": "",
            "year": np.nan,
            "text_type": "",
            "stage": "",
            "event": "",
            "full_text": "",
            "australia_passages": "",
            "n_chars_full": 0,
            "n_chars_passages": 0,
            "error": repr(e),
        }

rows = []
for _, row in tqdm(df_links_filtered.iterrows(), total=len(df_links_filtered)):
    rows.append(scrape_one_article(row))

df_mfa_all = pd.DataFrame(rows)
df_mfa_all.to_csv(
    "data/mfa_transcripts/mfa_article_scrape_all_attempts.csv",
    index=False,
    encoding="utf-8-sig",
)

df_mfa = df_mfa_all[
    (df_mfa_all["ok"] == True)
    & (df_mfa_all["n_chars_full"] > 80)
].copy()

df_mfa = df_mfa[
    df_mfa["year"].isna()
    | ((df_mfa["year"] >= START_YEAR) & (df_mfa["year"] <= END_YEAR))
].copy()

df_mfa = df_mfa.drop_duplicates(subset=["url"]).reset_index(drop=True)

df_mfa.to_csv(
    "data/mfa_transcripts/mfa_australia_passages.csv",
    index=False,
    encoding="utf-8-sig",
)

print("All attempts:", df_mfa_all.shape)
print("Valid Australia-related articles:", df_mfa.shape)
display(df_mfa[["date", "title", "keyword", "text_type", "event", "url"]].head(30))
print("Saved: data/mfa_transcripts/mfa_australia_passages.csv")

In [ ]:
# ============================================================
# 9. Validate and describe collected data
# ============================================================

csv_path = "data/mfa_transcripts/mfa_australia_passages.csv"
df_mfa = pd.read_csv(csv_path, encoding="utf-8-sig")

print("Shape:", df_mfa.shape)
print("Columns:", list(df_mfa.columns))

if df_mfa.empty:
    raise ValueError("The collected CSV is empty. Check debug files or reduce filtering.")

display(df_mfa.head())

print("\nYear distribution:")
display(df_mfa["year"].value_counts(dropna=False).sort_index())

print("\nText type distribution:")
display(df_mfa["text_type"].value_counts(dropna=False))

print("\nEvent distribution:")
display(df_mfa["event"].value_counts(dropna=False))

print("\nStage distribution:")
display(df_mfa["stage"].value_counts(dropna=False))

In [ ]:
# ============================================================
# 10. TOFI dictionary and scoring
# ============================================================

COMPOUND_TERMS = [
    "冷战思维", "干涉内政", "主权独立", "遏制打压",
    "反华势力", "域外势力", "五眼联盟", "印太战略",
    "人权问题", "新冠溯源", "华为禁令", "贸易制裁",
    "经济胁迫", "相互尊重", "不干涉", "互利共赢",
    "合作共赢", "挑衅行为", "对抗姿态", "零和博弈",
    "澳核潜艇", "核潜艇合作", "AUKUS",
]

for term in COMPOUND_TERMS:
    jieba.add_word(term, freq=1000)

TOFI_DICT = {
    "agency_attribution": [
        "蓄意", "故意", "刻意", "有意", "图谋", "妄图",
        "企图", "意图", "操弄", "推动", "主导", "幕后",
    ],
    "threat_characterization": [
        "威胁", "干涉", "遏制", "打压", "围堵", "制裁",
        "挑衅", "对抗", "敌视", "霸权", "单边", "胁迫",
        "恐吓", "冷战", "分裂", "颠覆", "渗透", "核扩散",
    ],
    "identity_contrast": [
        "冷战思维", "冷战逻辑", "意识形态偏见", "价值观",
        "反华", "涉华", "双重标准", "选择性", "偏见",
        "跟风", "附庸", "追随", "盲目", "短视",
    ],
    "victim_positioning": [
        "主权", "领土完整", "内政", "核心利益", "合法权益",
        "尊严", "无端", "无理", "污蔑", "抹黑",
        "造谣", "诽谤", "捏造", "歪曲",
    ],
    "cooperative_frame": [
        "合作", "互利", "共赢", "友好", "正常化", "重启",
        "对话", "沟通", "协商", "磋商", "务实", "稳定",
        "健康", "积极", "友谊", "伙伴关系",
    ],
}

def compute_tofi_scores(text):
    text = str(text)
    tokens = jieba.lcut(text)
    word_count = max(len(tokens), 1)

    scores = {}
    raw_threat_total = 0

    for dim, terms in TOFI_DICT.items():
        count = sum(text.count(term) for term in terms)
        scores[f"{dim}_raw"] = count
        scores[f"{dim}_norm"] = count / word_count * 1000

        if dim != "cooperative_frame":
            raw_threat_total += count

    coop_raw = scores["cooperative_frame_raw"]
    scores["tofi_raw"] = raw_threat_total
    scores["tofi_norm"] = raw_threat_total / word_count * 1000
    scores["coop_raw"] = coop_raw
    scores["coop_norm"] = coop_raw / word_count * 1000
    scores["tofi_net"] = scores["tofi_norm"] - scores["coop_norm"]
    scores["word_count"] = word_count

    return scores

score_rows = []
for _, row in df_mfa.iterrows():
    text = row.get("australia_passages", "")
    if not isinstance(text, str) or len(text.strip()) < 20:
        text = row.get("full_text", "")

    score_rows.append(compute_tofi_scores(text))

df_scores = pd.DataFrame(score_rows)
df_scored = pd.concat([df_mfa.reset_index(drop=True), df_scores], axis=1)

df_scored["date_dt"] = pd.to_datetime(df_scored["date"], errors="coerce")
df_scored["quarter"] = df_scored["date_dt"].dt.to_period("Q").astype(str)

df_scored.to_csv(
    "data/corpus/corpus_processed_tofi.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved: data/corpus/corpus_processed_tofi.csv")
display(df_scored[["date", "title", "tofi_norm", "coop_norm", "tofi_net", "word_count"]].head(20))

In [ ]:
# ============================================================
# 11. Aggregate TOFI and export summary tables
# ============================================================

tofi_quarterly = (
    df_scored
    .dropna(subset=["quarter"])
    .groupby("quarter")
    .agg(
        tofi_mean=("tofi_norm", "mean"),
        tofi_net_mean=("tofi_net", "mean"),
        coop_mean=("coop_norm", "mean"),
        doc_count=("title", "count"),
    )
    .reset_index()
)

tofi_quarterly.to_csv(
    "data/corpus/tofi_quarterly.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_table = (
    df_scored
    .groupby(["stage", "event", "text_type"], dropna=False)
    .agg(
        doc_count=("title", "count"),
        avg_tofi_norm=("tofi_norm", "mean"),
        avg_coop_norm=("coop_norm", "mean"),
        avg_tofi_net=("tofi_net", "mean"),
    )
    .reset_index()
    .sort_values(["stage", "doc_count"], ascending=[True, False])
)

summary_table.to_csv(
    "data/analysis/mfa_stage_event_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved: data/corpus/tofi_quarterly.csv")
display(tofi_quarterly.head(30))

print("Saved: data/analysis/mfa_stage_event_summary.csv")
display(summary_table.head(50))

In [ ]:
# ============================================================
# 12. Paper-ready visualisations and analytical tables
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from textwrap import fill

# ------------------------------------------------------------
# 12.1 Output folders
# ------------------------------------------------------------

os.makedirs("output/figures", exist_ok=True)
os.makedirs("data/analysis", exist_ok=True)

# ------------------------------------------------------------
# 12.2 Global academic plotting style
# ------------------------------------------------------------

plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"

# Academic muted palette
COLOR_MAIN = "#4C78A8"      # muted blue
COLOR_DARK = "#2F4B6C"      # dark blue
COLOR_LIGHT = "#A6BDD7"     # light blue
COLOR_MID = "#6E8FB3"       # medium blue
COLOR_GREY = "#B0B0B0"
COLOR_GRID = "#D9D9D9"
COLOR_THREAT = "#4C78A8"
COLOR_COOP = "#B7B7B7"

stage_order = [
    "Cooperative baseline period (2002–2016)",
    "Threat activation period (2017–2021)",
    "Repair with residual threat period (2022–2026)"
]

stage_short = {
    "Cooperative baseline period (2002–2016)": "Cooperative baseline\n(2002–2016)",
    "Threat activation period (2017–2021)": "Threat activation\n(2017–2021)",
    "Repair with residual threat period (2022–2026)": "Repair with residual threat\n(2022–2026)"
}

stage_palette = {
    "Cooperative baseline period (2002–2016)": "#A6BDD7",
    "Threat activation period (2017–2021)": "#6E8FB3",
    "Repair with residual threat period (2022–2026)": "#4C78A8"
}

def clean_axes(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, linestyle="--", alpha=0.25, color=COLOR_GRID)
    ax.set_axisbelow(True)

# ------------------------------------------------------------
# 12.3 Prepare analytical dataset
# ------------------------------------------------------------

df_plot = df_scored.copy()

df_plot["year"] = pd.to_numeric(df_plot["year"], errors="coerce")
df_plot = df_plot.dropna(subset=["year"]).copy()
df_plot["year"] = df_plot["year"].astype(int)

# Use the actual digitally available corpus period.
df_plot = df_plot[(df_plot["year"] >= 2002) & (df_plot["year"] <= 2026)].copy()

def assign_stage_by_year(year):
    if 2002 <= year <= 2016:
        return "Cooperative baseline period (2002–2016)"
    elif 2017 <= year <= 2021:
        return "Threat activation period (2017–2021)"
    elif 2022 <= year <= 2026:
        return "Repair with residual threat period (2022–2026)"
    else:
        return "Out of scope"

df_plot["stage_final"] = df_plot["year"].apply(assign_stage_by_year)
df_plot = df_plot[df_plot["stage_final"].isin(stage_order)].copy()

df_plot["stage_final"] = pd.Categorical(
    df_plot["stage_final"],
    categories=stage_order,
    ordered=True
)

# English issue labels
event_map = {
    "经贸摩擦/贸易争端": "Economic frictions / trade disputes",
    "双边关系修复/对话": "Bilateral repair / dialogue",
    "其他议题": "Other issues",
    "五眼联盟/印太安全": "Five Eyes / Indo-Pacific security",
    "AUKUS/核潜艇": "AUKUS / nuclear submarines",
    "人权/涉疆涉港涉台": "Human rights / Xinjiang / Hong Kong / Taiwan",
    "新冠溯源/疫情争议": "COVID-origin / pandemic disputes"
}

df_plot["event_en"] = df_plot["event"].map(event_map).fillna(df_plot["event"])

for col in ["tofi_norm", "coop_norm", "tofi_net", "word_count"]:
    if col in df_plot.columns:
        df_plot[col] = pd.to_numeric(df_plot[col], errors="coerce")

print("Dataset used for figures:", df_plot.shape)
display(df_plot[["date", "year", "title", "stage_final", "event_en", "tofi_norm", "coop_norm", "tofi_net"]].head(15))

# ============================================================
# Figure 1. Annual distribution of the corpus
# ============================================================

year_counts = (
    df_plot
    .groupby("year")
    .size()
    .reset_index(name="doc_count")
    .sort_values("year")
)

fig, ax = plt.subplots(figsize=(10.5, 4.8))

ax.bar(
    year_counts["year"],
    year_counts["doc_count"],
    color=COLOR_MAIN,
    edgecolor="white",
    linewidth=0.8
)

ax.axvline(2017, color="gray", linestyle="--", linewidth=1)
ax.axvline(2022, color="gray", linestyle="--", linewidth=1)

ymax = year_counts["doc_count"].max()

ax.text(2010, ymax * 0.92, "Cooperative baseline", ha="center", color="#444444")
ax.text(2019.5, ymax * 0.92, "Threat activation", ha="center", color="#444444")
ax.text(2024.1, ymax * 0.92, "Repair with residual threat", ha="center", color="#444444")

ax.set_title("Annual Distribution of Australia-related MFA Texts, 2002–2026")
ax.set_xlabel("Year")
ax.set_ylabel("Number of texts")

clean_axes(ax, "y")
plt.tight_layout()
plt.savefig("output/figures/Figure1_annual_distribution.png", bbox_inches="tight")
plt.show()

# ============================================================
# Figure 2. Corpus size across stages
# ============================================================

stage_counts = (
    df_plot
    .groupby("stage_final", observed=True)
    .size()
    .reindex(stage_order)
    .reset_index(name="doc_count")
)

stage_counts["stage_short"] = stage_counts["stage_final"].map(stage_short)

fig, ax = plt.subplots(figsize=(8.8, 4.8))

bars = ax.bar(
    stage_counts["stage_short"],
    stage_counts["doc_count"],
    color=COLOR_MAIN,
    edgecolor="white",
    linewidth=0.8
)

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{int(bar.get_height())}",
        ha="center",
        va="bottom",
        color="#333333"
    )

ax.set_title("Corpus Size across Historical Stages")
ax.set_xlabel("Stage")
ax.set_ylabel("Number of texts")

clean_axes(ax, "y")
plt.tight_layout()
plt.savefig("output/figures/Figure2_stage_counts.png", bbox_inches="tight")
plt.show()

# ============================================================
# Figure 3. TOFI and cooperation framing by stage
# This figure is useful because your argument is about coexistence.
# ============================================================

stage_scores = (
    df_plot
    .groupby("stage_final", observed=True)
    .agg(
        avg_tofi=("tofi_norm", "mean"),
        avg_coop=("coop_norm", "mean"),
        avg_net=("tofi_net", "mean"),
        median_tofi=("tofi_norm", "median"),
        median_coop=("coop_norm", "median"),
        doc_count=("title", "count")
    )
    .reindex(stage_order)
    .reset_index()
)

stage_scores["stage_short"] = stage_scores["stage_final"].map(stage_short)

x = np.arange(len(stage_scores))
width = 0.34

fig, ax = plt.subplots(figsize=(9.5, 4.8))

bars1 = ax.bar(
    x - width / 2,
    stage_scores["avg_tofi"],
    width,
    label="Threat framing (TOFI)",
    color=COLOR_DARK,
    edgecolor="white",
    linewidth=0.8
)

bars2 = ax.bar(
    x + width / 2,
    stage_scores["avg_coop"],
    width,
    label="Cooperation framing",
    color=COLOR_GREY,
    edgecolor="white",
    linewidth=0.8
)

ax.set_title("Threat and Cooperation Framing across Stages")
ax.set_xlabel("Stage")
ax.set_ylabel("Average standardized score")
ax.set_xticks(x)
ax.set_xticklabels(stage_scores["stage_short"])
ax.legend(frameon=False)

clean_axes(ax, "y")
plt.tight_layout()
plt.savefig("output/figures/Figure3_tofi_cooperation_by_stage.png", bbox_inches="tight")
plt.show()

# ============================================================
# Figure 4. Average net framing score by stage
# Negative values mean cooperation framing exceeds threat framing.
# ============================================================

fig, ax = plt.subplots(figsize=(8.8, 4.8))

bars = ax.bar(
    stage_scores["stage_short"],
    stage_scores["avg_net"],
    color=COLOR_MAIN,
    edgecolor="white",
    linewidth=0.8
)

ax.axhline(0, color="black", linewidth=1)

for bar, val in zip(bars, stage_scores["avg_net"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val - 1.0 if val < 0 else val + 0.5,
        f"{val:.2f}",
        ha="center",
        va="top" if val < 0 else "bottom",
        color="#333333"
    )

ax.set_title("Average Net Threat Framing by Stage")
ax.set_xlabel("Stage")
ax.set_ylabel("Net framing score\n(Threat framing − Cooperation framing)")

clean_axes(ax, None)
plt.tight_layout()
plt.savefig("output/figures/Figure4_net_framing_by_stage.png", bbox_inches="tight")
plt.show()

# ============================================================
# Figure 5. Annual net framing trend
# ============================================================

year_scores = (
    df_plot
    .groupby("year")
    .agg(
        avg_tofi=("tofi_norm", "mean"),
        avg_coop=("coop_norm", "mean"),
        avg_net=("tofi_net", "mean"),
        doc_count=("title", "count")
    )
    .reset_index()
    .sort_values("year")
)

year_scores["net_roll3"] = (
    year_scores["avg_net"]
    .rolling(window=3, min_periods=1)
    .mean()
)

fig, ax = plt.subplots(figsize=(10.5, 4.8))

ax.plot(
    year_scores["year"],
    year_scores["net_roll3"],
    color=COLOR_DARK,
    linewidth=2.2,
    marker="o",
    markersize=4
)

ax.axhline(0, color="black", linewidth=1)
ax.axvline(2017, color="gray", linestyle="--", linewidth=1)
ax.axvline(2022, color="gray", linestyle="--", linewidth=1)

ax.set_title("Three-year Rolling Trend of Net Threat Framing")
ax.set_xlabel("Year")
ax.set_ylabel("Net framing score\n(Threat framing − Cooperation framing)")

clean_axes(ax, "y")
plt.tight_layout()
plt.savefig("output/figures/Figure5_annual_net_trend.png", bbox_inches="tight")
plt.show()

# ============================================================
# Figure 6. Issue composition across stages
# Core figure for showing issue-specific discourse structure.
# ============================================================

issue_stage_pct = pd.crosstab(
    df_plot["event_en"],
    df_plot["stage_final"],
    normalize="columns"
).reindex(columns=stage_order) * 100

# Order issues by overall frequency, high to low
issue_order = (
    df_plot["event_en"]
    .value_counts()
    .index
    .tolist()
)

issue_stage_pct = issue_stage_pct.reindex(issue_order)

heatmap_cols = [stage_short[s] for s in issue_stage_pct.columns]

fig, ax = plt.subplots(figsize=(10.2, 5.8))

im = ax.imshow(
    issue_stage_pct.values,
    cmap="Blues",
    aspect="auto"
)

ax.set_xticks(np.arange(len(heatmap_cols)))
ax.set_xticklabels(heatmap_cols)

ax.set_yticks(np.arange(len(issue_stage_pct.index)))
ax.set_yticklabels([fill(i, 32) for i in issue_stage_pct.index])

for i in range(issue_stage_pct.shape[0]):
    for j in range(issue_stage_pct.shape[1]):
        val = issue_stage_pct.iloc[i, j]
        ax.text(
            j,
            i,
            f"{val:.1f}",
            ha="center",
            va="center",
            color="white" if val > 25 else "#222222",
            fontsize=9
        )

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Share within stage (%)")

ax.set_title("Issue Composition across Stages")
ax.set_xlabel("Stage")
ax.set_ylabel("Issue category")

plt.tight_layout()
plt.savefig("output/figures/Figure6_issue_composition_heatmap.png", bbox_inches="tight")
plt.show()

# ============================================================
# Figure 7. Share of high-threat-framing texts by stage
# This is one of the strongest figures for your revised argument.
# ============================================================

threshold = df_plot["tofi_norm"].quantile(0.75)

df_plot["high_tofi"] = (df_plot["tofi_norm"] >= threshold).astype(int)

high_tofi_stage = (
    df_plot
    .groupby("stage_final", observed=True)
    .agg(
        high_tofi_share=("high_tofi", "mean"),
        doc_count=("title", "count")
    )
    .reindex(stage_order)
    .reset_index()
)

high_tofi_stage["high_tofi_share"] = high_tofi_stage["high_tofi_share"] * 100
high_tofi_stage["stage_short"] = high_tofi_stage["stage_final"].map(stage_short)

fig, ax = plt.subplots(figsize=(8.8, 4.8))

bars = ax.bar(
    high_tofi_stage["stage_short"],
    high_tofi_stage["high_tofi_share"],
    color=COLOR_MAIN,
    edgecolor="white",
    linewidth=0.8
)

for bar, val in zip(bars, high_tofi_stage["high_tofi_share"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.8,
        f"{val:.1f}%",
        ha="center",
        color="#333333"
    )

ax.set_title("Share of High-Threat-Framing Texts by Stage")
ax.set_xlabel("Stage")
ax.set_ylabel("Percentage of texts")

clean_axes(ax, "y")
plt.tight_layout()
plt.savefig("output/figures/Figure7_high_tofi_share.png", bbox_inches="tight")
plt.show()

# ============================================================
# Figure 8. Average TOFI by issue category
# This helps identify where threat framing is concentrated.
# ============================================================

issue_tofi = (
    df_plot
    .groupby("event_en")
    .agg(
        avg_tofi=("tofi_norm", "mean"),
        avg_coop=("coop_norm", "mean"),
        doc_count=("title", "count")
    )
    .reset_index()
)

# Keep issue categories with at least 3 texts to avoid unstable averages.
issue_tofi = issue_tofi[issue_tofi["doc_count"] >= 3].copy()
issue_tofi = issue_tofi.sort_values("avg_tofi", ascending=True)

fig, ax = plt.subplots(figsize=(9.5, 5.2))

bars = ax.barh(
    issue_tofi["event_en"].apply(lambda x: fill(x, 35)),
    issue_tofi["avg_tofi"],
    color=COLOR_MAIN,
    edgecolor="white",
    linewidth=0.8
)

for bar, val in zip(bars, issue_tofi["avg_tofi"]):
    ax.text(
        val + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}",
        va="center",
        color="#333333"
    )

ax.set_title("Average Threat Framing Score by Issue Category")
ax.set_xlabel("Average TOFI score")
ax.set_ylabel("Issue category")

clean_axes(ax, "x")
plt.tight_layout()
plt.savefig("output/figures/Figure8_issue_average_tofi.png", bbox_inches="tight")
plt.show()

# ============================================================
# Figure 9. Issue-stage high-TOFI share
# More specific than general stage averages.
# ============================================================

issue_stage_high = (
    df_plot
    .groupby(["event_en", "stage_final"], observed=True)
    .agg(
        high_tofi_share=("high_tofi", "mean"),
        doc_count=("title", "count")
    )
    .reset_index()
)

# Convert to percentage
issue_stage_high["high_tofi_share"] = issue_stage_high["high_tofi_share"] * 100

# Keep issue-stage cells with at least 3 observations
issue_stage_high_filtered = issue_stage_high[issue_stage_high["doc_count"] >= 3].copy()

pivot_high = issue_stage_high_filtered.pivot(
    index="event_en",
    columns="stage_final",
    values="high_tofi_share"
).reindex(columns=stage_order)

# Reorder rows using issue_order
pivot_high = pivot_high.reindex(issue_order)
pivot_high = pivot_high.dropna(how="all")

fig, ax = plt.subplots(figsize=(10.2, 5.8))

im = ax.imshow(
    pivot_high.fillna(0).values,
    cmap="Blues",
    aspect="auto",
    vmin=0,
    vmax=100
)

ax.set_xticks(np.arange(len(stage_order)))
ax.set_xticklabels([stage_short[s] for s in stage_order])

ax.set_yticks(np.arange(len(pivot_high.index)))
ax.set_yticklabels([fill(i, 32) for i in pivot_high.index])

for i in range(pivot_high.shape[0]):
    for j in range(pivot_high.shape[1]):
        val = pivot_high.iloc[i, j]
        if pd.isna(val):
            txt = "NA"
            color = "#777777"
        else:
            txt = f"{val:.1f}"
            color = "white" if val > 50 else "#222222"

        ax.text(
            j,
            i,
            txt,
            ha="center",
            va="center",
            color=color,
            fontsize=9
        )

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("High-TOFI share (%)")

ax.set_title("High-Threat-Framing Share by Issue and Stage")
ax.set_xlabel("Stage")
ax.set_ylabel("Issue category")

plt.tight_layout()
plt.savefig("output/figures/Figure9_issue_stage_high_tofi.png", bbox_inches="tight")
plt.show()

# ============================================================
# Top high-TOFI texts for qualitative discussion
# ============================================================

top_tofi_docs = (
    df_plot
    .sort_values("tofi_norm", ascending=False)
    [["date", "year", "title", "stage_final", "event_en", "tofi_norm", "coop_norm", "tofi_net", "url"]]
    .head(20)
)

print("Top high-TOFI documents for qualitative discussion:")
display(top_tofi_docs)

# ============================================================
# Save analytical tables
# ============================================================

year_counts.to_csv(
    "data/analysis/year_counts_en.csv",
    index=False,
    encoding="utf-8-sig"
)

stage_counts.to_csv(
    "data/analysis/stage_counts_en.csv",
    index=False,
    encoding="utf-8-sig"
)

stage_scores.to_csv(
    "data/analysis/stage_scores_en.csv",
    index=False,
    encoding="utf-8-sig"
)

year_scores.to_csv(
    "data/analysis/year_scores_en.csv",
    index=False,
    encoding="utf-8-sig"
)

issue_stage_pct.to_csv(
    "data/analysis/issue_stage_pct_en.csv",
    encoding="utf-8-sig"
)

high_tofi_stage.to_csv(
    "data/analysis/high_tofi_stage_en.csv",
    index=False,
    encoding="utf-8-sig"
)

issue_tofi.to_csv(
    "data/analysis/issue_average_tofi_en.csv",
    index=False,
    encoding="utf-8-sig"
)

issue_stage_high_filtered.to_csv(
    "data/analysis/issue_stage_high_tofi_en.csv",
    index=False,
    encoding="utf-8-sig"
)

top_tofi_docs.to_csv(
    "data/analysis/top_tofi_docs_en.csv",
    index=False,
    encoding="utf-8-sig"
)

df_plot.to_csv(
    "data/analysis/figure_dataset_2002_2026_en.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Academic-style paper-ready figures saved to: output/figures/")
print("Analytical tables saved to: data/analysis/")

In [ ]:
# ============================================================
# Figure X. Distribution of document-level TOFI scores by stage
# ============================================================

fig, ax = plt.subplots(figsize=(9.2, 5.2))

box_data = [
    df_plot.loc[df_plot["stage_final"] == stage, "tofi_norm"].dropna()
    for stage in stage_order
]

box = ax.boxplot(
    box_data,
    patch_artist=True,
    widths=0.55,
    showmeans=True,
    meanline=False,
    medianprops=dict(color="#222222", linewidth=1.5),
    meanprops=dict(marker="o", markerfacecolor="#222222", markeredgecolor="#222222", markersize=5),
    boxprops=dict(linewidth=1),
    whiskerprops=dict(linewidth=1),
    capprops=dict(linewidth=1),
    flierprops=dict(marker="o", markerfacecolor="#777777", markeredgecolor="#777777", markersize=3, alpha=0.45)
)

# Academic blue-grey palette
box_colors = ["#A6BDD7", "#6E8FB3", "#4C78A8"]

for patch, color in zip(box["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

ax.set_xticks(np.arange(1, len(stage_order) + 1))
ax.set_xticklabels([stage_short[s] for s in stage_order])

ax.set_title("Distribution of Document-level TOFI Scores by Stage")
ax.set_xlabel("Stage")
ax.set_ylabel("Document-level TOFI score")

clean_axes(ax, "y")

plt.tight_layout()
plt.savefig(
    "output/figures/FigureX_tofi_distribution_by_stage.png",
    bbox_inches="tight"
)
plt.show()

In [ ]:
# ============================================================
# 13. Export paper-ready outputs
# ============================================================

import os
import zipfile
import pandas as pd

os.makedirs("output/final_outputs", exist_ok=True)

# ------------------------------------------------------------
# 13.1 Export Excel workbook
# ------------------------------------------------------------

excel_path = "output/final_outputs/mfa_australia_tofi_analysis_2002_2026.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_plot.to_excel(writer, sheet_name="figure_dataset", index=False)
    df_scored.to_excel(writer, sheet_name="full_tofi_scored", index=False)

    year_counts.to_excel(writer, sheet_name="year_counts", index=False)
    stage_counts.to_excel(writer, sheet_name="stage_counts", index=False)
    stage_scores.to_excel(writer, sheet_name="stage_scores", index=False)
    year_scores.to_excel(writer, sheet_name="year_scores", index=False)

    issue_stage_pct.to_excel(writer, sheet_name="issue_stage_pct")
    high_tofi_stage.to_excel(writer, sheet_name="high_tofi_stage", index=False)
    issue_tofi.to_excel(writer, sheet_name="issue_average_tofi", index=False)
    issue_stage_high_filtered.to_excel(writer, sheet_name="issue_stage_high_tofi", index=False)

    top_tofi_docs.to_excel(writer, sheet_name="top_high_tofi_docs", index=False)

print("Excel workbook saved:")
print(excel_path)

# ------------------------------------------------------------
# 13.2 Zip all figures
# ------------------------------------------------------------

fig_zip_path = "output/final_outputs/paper_ready_figures_2002_2026.zip"

with zipfile.ZipFile(fig_zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk("output/figures"):
        for file in files:
            if file.endswith(".png"):
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, "output/figures")
                zipf.write(file_path, arcname)

print("Figure zip saved:")
print(fig_zip_path)

# ------------------------------------------------------------
# 13.3 Zip analytical CSV tables
# ------------------------------------------------------------

csv_zip_path = "output/final_outputs/paper_ready_tables_2002_2026.zip"

csv_files = [
    "data/analysis/figure_dataset_2002_2026_en.csv",
    "data/analysis/year_counts_en.csv",
    "data/analysis/stage_counts_en.csv",
    "data/analysis/stage_scores_en.csv",
    "data/analysis/year_scores_en.csv",
    "data/analysis/issue_stage_pct_en.csv",
    "data/analysis/high_tofi_stage_en.csv",
    "data/analysis/issue_average_tofi_en.csv",
    "data/analysis/issue_stage_high_tofi_en.csv",
    "data/analysis/top_tofi_docs_en.csv",
]

with zipfile.ZipFile(csv_zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in csv_files:
        if os.path.exists(file_path):
            zipf.write(file_path, os.path.basename(file_path))

print("CSV table zip saved:")
print(csv_zip_path)

# ------------------------------------------------------------
# 13.4 Optional download in Google Colab
# ------------------------------------------------------------

try:
    from google.colab import files

    files.download(excel_path)
    files.download(fig_zip_path)
    files.download(csv_zip_path)

except Exception:
    print("Download skipped. If you are not in Colab, files are saved locally.")

## Methods note

Because the MFA search page is dynamically rendered, ordinary `requests` scraping cannot reliably extract all search results from the initial HTML. This notebook uses Playwright to simulate a real browser, wait for JavaScript-rendered results, scroll the page, extract article links from the rendered DOM, and crawl result pages sequentially. It also listens to network responses and attempts to recover article URLs from JSON or text responses. To reduce omission, the crawler searches multiple China–Australia-related keywords and deduplicates links by URL before downloading article bodies.